In [1]:
from llama_cpp.llama import Llama, LlamaGrammar
import httpx
from llama_index.core.node_parser import SentenceSplitter

from llama_cpp.llama import LlamaGrammar
import numpy as np
import pandas as pd
import torch
# from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.llms.llama_cpp.llama_utils import (
    messages_to_prompt,
    completion_to_prompt,
)
from llama_index.core import Settings
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
import textwrap
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine

/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_url" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_path" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.read

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-14B-Instruct")

llm = LlamaCPP(
    # You can pass in the URL to a GGML model to download it automatically
    # optionally, you can set the path to a pre-downloaded model instead of model_url
    # model_path="/hf_cache/models--NousResearch--Hermes-3-Llama-3.1-8B-GGUF/snapshots/307a5dfb59aa38d88b6cfd32f44b8ad7c1da9fb8/Hermes-3-Llama-3.1-8B.Q5_K_M.gguf",
    # model_url="https://huggingface.co/bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF/blob/main/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
    model_path="/hf_cache/models--Qwen--Qwen2.5-Coder-14B-Instruct-GGUF/snapshots/3717cc660f02c1a47932785af057b3c22fc43f3a/qwen2.5-coder-14b-instruct-q6_k-00001-of-00002.gguf",
    temperature=0.1,
    max_new_tokens=4096,
    context_window=24000,
    generate_kwargs={
        "repeat_penalty": 1.1,
        "top_k": 0,
        "top_p": 0
    },
    model_kwargs={
        "n_gpu_layers": -1,
        # "grammar": grammar
                 },
    # messages_to_prompt=messages_to_prompt,
    # completion_to_prompt=completion_to_prompt,
    verbose=True,
)

Settings.llm = llm

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

llama_model_loader: additional 1 GGUFs metadata loaded.
llama_model_loader: loaded meta data with 29 key-value pairs and 579 tensors from /hf_cache/models--Qwen--Qwen2.5-Coder-14B-Instruct-GGUF/snapshots/3717cc660f02c1a47932785af057b3c22fc43f3a/qwen2.5-coder-14b-instruct-q6_k-00001-of-00002.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 Coder 14B Instruct AWQ
llama_model_loader: - kv   3:                           general.finetune str              = Instruct-AWQ
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5-Coder
llama_model_loader: - kv   5:                   

In [3]:
vector_store = PGVectorStore.from_params(
    database='grover',
    host='postgres',
    password='grover',
    port=5432,
    user='grover',
    table_name="crow_text",
    embed_dim=384,  
    hnsw_kwargs={
        "hnsw_m": 16,
        "hnsw_ef_construction": 64,
        "hnsw_ef_search": 40,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [4]:
text = SimpleDirectoryReader("/notebooks/llm/crow_rag/crowdocs/text/").load_data()

In [5]:
type(text)

list

In [6]:
# per-index
index = VectorStoreIndex.from_documents(
    text, storage_context=storage_context,
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=50)], show_progress=True
)


# index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

Generating embeddings: 100%|██████████| 33/33 [00:04<00:00,  7.88it/s]


In [16]:
print(index.as_query_engine().query("Write a crow function to generate rhythmic patterns using sequins."))

Llama.generate: 25 prefix-match hit, remaining 620 prompt tokens to eval

llama_print_timings:        load time =    1980.73 ms
llama_print_timings:      sample time =    4116.42 ms /   423 runs   (    9.73 ms per token,   102.76 tokens per second)
llama_print_timings: prompt eval time =    3726.77 ms /   620 tokens (    6.01 ms per token,   166.36 tokens per second)
llama_print_timings:        eval time =   23482.22 ms /   422 runs   (   55.65 ms per token,    17.97 tokens per second)
llama_print_timings:       total time =   31748.54 ms /  1042 tokens


 To generate rhythmic patterns using sequins in Crow, you can create a function that utilizes the `sequins` library to cycle through different rhythm values. Here's an example of how you might write such a function:

```lua
-- Define a function to generate rhythmic patterns
function generate_rhythmic_pattern(pattern)
    -- Create a sequins object with the given pattern
    local seq = sequins(pattern)

    -- Return a function that will be called by the metro event
    return function(count)
        -- Get the next value from the sequins
        local rhythm_value = seq()

        -- Use the rhythm value to trigger events or control other parameters
        print("Rhythm Value: " .. rhythm_value)

        -- Example: Trigger an event based on the rhythm value
        if rhythm_value == 1 then
            -- Do something for a quarter note
        elseif rhythm_value == 2 then
            -- Do something for an eighth note
        elseif rhythm_value == 4 then
            -- Do somethi

In [11]:
a = llm.complete("Write a python program to demonstrate class composition in object oriented programming.")

Llama.generate: 12 prefix-match hit, remaining 1 prompt tokens to eval

llama_print_timings:        load time =    1980.73 ms
llama_print_timings:      sample time =    4096.43 ms /   426 runs   (    9.62 ms per token,   103.99 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =   23726.67 ms /   426 runs   (   55.70 ms per token,    17.95 tokens per second)
llama_print_timings:       total time =   28250.50 ms /   426 tokens


In [14]:
print(a.text)

 Class composition is a design principle where one class contains another class as its member variable.

```python
# Define the Engine class
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower

    def start(self):
        print("Engine started with {} horsepower".format(self.horsepower))

# Define the Car class that uses composition to include an Engine
class Car:
    def __init__(self, make, model, engine):
        self.make = make
        self.model = model
        self.engine = engine  # Composition: Car has an Engine

    def display_info(self):
        print("Car Make: {}, Model: {}".format(self.make, self.model))
        self.engine.start()

# Create an instance of Engine
engine = Engine(200)

# Create an instance of Car using the created Engine
car = Car("Toyota", "Corolla", engine)

# Display information about the car and start its engine
car.display_info()
```

In this example, the `Car` class is composed of an `Engine` object. The `Car` cla